# BaitBreak – Scam Message Detection: Training Notebook
**Pipeline:** `dataset.csv` → group-aware split → TF-IDF (word + character n-grams) → Logistic Regression vs Multinomial Naive Bayes → pick best by cross-validated F1 → evaluate on hold-out **and** on a separate hand-written unseen Hindi/Hinglish/English set → save `model.pkl` + `vectorizer.pkl`.

Labels: **0 = Legitimate, 1 = Scam**. Run `python build_dataset.py` first if `dataset.csv` is missing.

In [1]:
import json, warnings, joblib, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from preprocess import clean_text, detect_language
warnings.filterwarnings("ignore")
SEED = 42
df = pd.read_csv("dataset.csv")
df["clean"] = df["message"].map(clean_text)      # same cleaner the API uses
print(df.shape)
df.head()

(5780, 6)


,message,clean,label,language,source,group
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,0,english,uci_sms,row_0
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni,0,english,uci_sms,row_1
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,1,english,uci_sms,row_2
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,0,english,uci_sms,row_3
4,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...,0,english,uci_sms,row_4


## 1. Dataset overview

In [2]:
print(df.groupby(["language", "label"]).size().unstack(fill_value=0).rename(columns={0: "legit", 1: "scam"}))
print()
print(df.source.value_counts())

label     legit  scam
language             
english    4610   724
hindi       107   112
hinglish    117   110

source
uci_sms           5106
synthetic_seed     674
Name: count, dtype: int64


## 2. Train / test split
Synthetic seed messages come from templates, so a plain random split would put near-identical sentences in both train and test and inflate scores. We split with `GroupShuffleSplit` so **every template family stays entirely on one side** (UCI rows are their own groups).

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(df, df.label, groups=df.group))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
print("train", train.shape, "test", test.shape)
print(test.groupby(["language", "label"]).size().unstack(fill_value=0))

train (4626, 6) test (1154, 6)
label       0    1
language          
english   925  145
hindi      19   20
hinglish   19   26


## 3. TF-IDF + model comparison
The vectorizer combines **word 1-2-grams** (meaning) and **character 2-5-grams** (robust to Hinglish spelling variants such as *karein/karen/karo* and to Devanagari). Hyper-parameters are chosen by 5-fold `GroupKFold` cross-validation (F1 of the scam class) on the training split only.

In [4]:
def make_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), token_pattern=r"\S+", sublinear_tf=True, min_df=1)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)),
    ])

candidates = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
                            {"clf__C": [1, 5, 20]}),
    "Multinomial Naive Bayes": (MultinomialNB(), {"clf__alpha": [0.05, 0.1, 0.5, 1.0]}),
}
cv = GroupKFold(n_splits=5)
searches = {}
for name, (clf, grid) in candidates.items():
    pipe = Pipeline([("vec", make_vectorizer()), ("clf", clf)])
    gs = GridSearchCV(pipe, grid, scoring="f1", cv=cv, n_jobs=-1)
    gs.fit(train.clean, train.label, groups=train.group)
    searches[name] = gs
    print(f"{name:25s} best params {gs.best_params_}  CV F1 = {gs.best_score_:.4f}")

Logistic Regression       best params {'clf__C': 20}  CV F1 = 0.8980


Multinomial Naive Bayes   best params {'clf__alpha': 0.05}  CV F1 = 0.8777


## 4. Hold-out evaluation (accuracy, precision, recall, F1, confusion matrix)

In [5]:
def metrics(y, p):
    return dict(accuracy=accuracy_score(y, p), precision=precision_score(y, p, zero_division=0),
                recall=recall_score(y, p, zero_division=0), f1=f1_score(y, p, zero_division=0),
                confusion_matrix=confusion_matrix(y, p, labels=[0, 1]).tolist())

holdout = {}
for name, gs in searches.items():
    pred = gs.predict(test.clean)
    holdout[name] = metrics(test.label, pred)
    holdout[name].update(cv_f1=gs.best_score_, best_params=gs.best_params_)
    print("=" * 60, "\n", name, "\n", classification_report(test.label, pred, target_names=["Legitimate", "Scam"], digits=4))
pd.DataFrame(holdout).T[["accuracy", "precision", "recall", "f1", "cv_f1"]].round(4)

 Logistic Regression 
               precision    recall  f1-score   support

  Legitimate     0.9915    0.9699    0.9806       963
        Scam     0.8632    0.9581    0.9082       191

    accuracy                         0.9679      1154
   macro avg     0.9274    0.9640    0.9444      1154
weighted avg     0.9703    0.9679    0.9686      1154



 Multinomial Naive Bayes 
               precision    recall  f1-score   support

  Legitimate     0.9902    0.9450    0.9671       963
        Scam     0.7745    0.9529    0.8545       191

    accuracy                         0.9463      1154
   macro avg     0.8823    0.9489    0.9108      1154
weighted avg     0.9545    0.9463    0.9484      1154



,accuracy,precision,recall,f1,cv_f1
Logistic Regression,0.967938,0.863208,0.958115,0.908189,0.898028
Multinomial Naive Bayes,0.946274,0.774468,0.95288,0.85446,0.877724


In [6]:
best_name = max(searches, key=lambda n: searches[n].best_score_)   # chosen on CV only, never on the test set
print("Selected model:", best_name)
best_pred = searches[best_name].predict(test.clean)

# per-language and per-source hold-out performance of the selected model
rows = []
for lang, g in test.groupby("language"):
    p = best_pred[g.index]
    rows.append(dict(slice=f"language={lang}", n=len(g), **{k: v for k, v in metrics(g.label, p).items() if k != "confusion_matrix"}))
uci = test[test.source == "uci_sms"]
rows.append(dict(slice="source=uci_sms (real English SMS only)", n=len(uci),
                 **{k: v for k, v in metrics(uci.label, best_pred[uci.index]).items() if k != "confusion_matrix"}))
per_slice = pd.DataFrame(rows).round(4); per_slice

Selected model: Logistic Regression


,slice,n,accuracy,precision,recall,f1
0,language=english,1070,0.9748,0.8782,0.9448,0.9103
1,language=hindi,39,0.7949,0.7143,1.0000,0.8333
2,language=hinglish,45,0.9556,0.9286,1.0000,0.9630
3,source=uci_sms (real English SMS only),1024,0.9902,0.9836,0.9375,0.9600


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (name, gs) in zip(axes, searches.items()):
    cm = np.array(holdout[name]["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["Legit", "Scam"]); ax.set_yticks([0, 1], ["Legit", "Scam"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name, fontsize=10)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
plt.tight_layout(); plt.savefig("reports/confusion_holdout.png", dpi=160); plt.show()

## 5. Final model
The selected configuration is **re-fitted on all of `dataset.csv`** (train + hold-out) so the shipped model uses as much Hindi/Hinglish data as possible. The hold-out numbers above come from the train-only model; the truly independent check of the *shipped* model is the unseen set in section 6.

In [8]:
best_pipe = searches[best_name].best_estimator_
vec = clone(best_pipe.named_steps["vec"]); clf = clone(best_pipe.named_steps["clf"])
X_all = vec.fit_transform(df.clean)
clf.fit(X_all, df.label)
joblib.dump(clf, "model.pkl"); joblib.dump(vec, "vectorizer.pkl")
print("saved model.pkl / vectorizer.pkl ->", type(clf).__name__, "| features:", X_all.shape[1])

saved model.pkl / vectorizer.pkl -> LogisticRegression | features: 81004


## 6. Unseen Hindi / Hinglish / English messages
`data/unseen_test.csv` – 72 hand-written messages (12 scam + 12 legitimate per language) that are **not** in `dataset.csv` and were never used for training or model selection. It includes tricky legitimate bank/OTP messages ("Do not share OTP") that resemble scams.

In [9]:
un = pd.read_csv("data/unseen_test.csv")
m = joblib.load("model.pkl"); v = joblib.load("vectorizer.pkl")
proba = m.predict_proba(v.transform(un.message.map(clean_text)))
un["pred"] = proba.argmax(1); un["confidence"] = proba.max(1).round(3)
unseen = {"overall": metrics(un.label, un.pred), "per_language": {}}
for lang, g in un.groupby("language"):
    unseen["per_language"][lang] = {k: v_ for k, v_ in metrics(g.label, g.pred).items() if k != "confusion_matrix"}
    unseen["per_language"][lang]["n"] = len(g)
print({k: round(v_, 4) for k, v_ in unseen["overall"].items() if k != "confusion_matrix"})
pd.DataFrame(unseen["per_language"]).T.round(4)

{'accuracy': 0.9028, 'precision': 0.8919, 'recall': 0.9167, 'f1': 0.9041}


,accuracy,precision,recall,f1,n
english,0.8333,0.8333,0.8333,0.8333,24.0
hindi,0.9583,0.9231,1.0000,0.9600,24.0
hinglish,0.9167,0.9167,0.9167,0.9167,24.0


In [10]:
wrong = un[un.label != un.pred][["language", "label", "pred", "confidence", "message"]]
unseen["errors"] = wrong.assign(message=wrong.message).to_dict("records")
print(f"{len(wrong)} / {len(un)} unseen messages misclassified")
wrong

7 / 72 unseen messages misclassified


,language,label,pred,confidence,message
7,english,1,0,0.805,"Hi dear, I am stuck at the airport and my card..."
8,english,1,0,0.504,Your WhatsApp will be deactivated in 1 hour. F...
13,english,0,1,0.553,Your OTP for Swiggy login is 482913. Do not sh...
14,english,0,1,0.857,"Rs 1,250 spent on your HDFC card ending 4521 a..."
31,hinglish,1,0,0.786,Aapka WhatsApp 1 ghante mein band ho jayega. A...
38,hinglish,0,1,0.899,Aapke HDFC card ending 4521 se Big Bazaar mein...
62,hindi,0,1,0.950,आपके एचडीएफसी कार्ड (4521) से 02 नवंबर को बिग ...


In [11]:
cm = np.array(unseen["overall"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4, 3.6)); ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["Legit", "Scam"]); ax.set_yticks([0, 1], ["Legit", "Scam"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Unseen set (n=72)", fontsize=10)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
plt.tight_layout(); plt.savefig("reports/confusion_unseen.png", dpi=160); plt.show()

## 7. Save metrics (used to build `evaluation_report.pdf`) and smoke-test

In [12]:
out = dict(
    selected_model=best_name,
    n_total=len(df), n_train=len(train), n_test=len(test),
    dataset_by_language=df.groupby(["language", "label"]).size().unstack(fill_value=0).rename(columns={0: "legit", 1: "scam"}).to_dict("index"),
    dataset_by_source=df.source.value_counts().to_dict(),
    holdout=holdout, holdout_per_slice=per_slice.to_dict("records"), unseen=unseen)
json.dump(out, open("reports/metrics.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=float)

for msg in ["Your bank account will be blocked. Share OTP immediately.",
            "आपका खाता ब्लॉक हो जाएगा, तुरंत OTP भेजें",
            "Bhai kal 6 baje milte hain, late mat hona"]:
    p = m.predict_proba(v.transform([clean_text(msg)]))[0]
    print(("scam" if p[1] >= .5 else "legitimate"), round(float(p.max()), 3), "|", msg)

scam 0.902 | Your bank account will be blocked. Share OTP immediately.
scam 0.815 | आपका खाता ब्लॉक हो जाएगा, तुरंत OTP भेजें
legitimate 0.994 | Bhai kal 6 baje milte hain, late mat hona
